In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys, time, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
sys.path.insert(0, "/content/drive/MyDrive/esci/code")
import cross_encoder as C

BASE = "/content/drive/MyDrive/esci"
best = pd.read_csv(f"{BASE}/results/s8_best_epochs.csv")
val = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet").query("part == 'val'")
val["q"] = [C.query_to_text(q) for q in val["query"]]
val["t"] = [C.product_to_text(C.esci_row_to_product(r), "full_bullets") for r in val.to_dict("records")]  # longest format = worst case

rng = np.random.default_rng(0)
qids = rng.choice(val["query_id"].unique(), 50, replace=False)
pool = val["t"].tolist()
batches = []                                   # 50 queries x exactly 30 candidates
for qid in qids:
    g = val[val.query_id == qid]
    texts = g["t"].tolist()[:30]
    texts += list(rng.choice(pool, 30 - len(texts), replace=False))
    batches.append((g["q"].iloc[0], texts))

def latency(path, device, n_threads=None):
    if n_threads: torch.set_num_threads(n_threads)
    tok = AutoTokenizer.from_pretrained(path)
    m = AutoModelForSequenceClassification.from_pretrained(path).to(device).eval()
    times = []
    with torch.no_grad():
        for i, (q, texts) in enumerate(batches[:5] + batches):          # first 5 = warm-up
            t0 = time.perf_counter()
            enc = tok([q] * len(texts), texts, truncation=True, max_length=256, padding=True, return_tensors="pt").to(device)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
                m(**enc).logits
            if device == "cuda": torch.cuda.synchronize()
            if i >= 5: times.append((time.perf_counter() - t0) * 1000)
    del m; torch.cuda.empty_cache()
    return np.median(times), np.percentile(times, 95)

rows = []
for short in ["minilm_l6", "minilm_l12", "distilbert", "bert_base"]:
    r = best[(best["short"] == short) & (best["qin"] == "q_focused_cat") & (best["seed"] == 42)].iloc[0]
    path = f"{BASE}/models/{short}__q_focused_cat/epoch_{int(r['epoch'])}"
    g_med, g_p95 = latency(path, "cuda")
    c_med, c_p95 = latency(path, "cpu")
    params = sum(p.numel() for p in AutoModelForSequenceClassification.from_pretrained(path).parameters()) / 1e6
    rows.append({"model": short, "params_M": round(params, 1), "val_ndcg": round(r["ndcg@10"], 4),
                 "gpu_ms_median": round(g_med, 1), "gpu_ms_p95": round(g_p95, 1),
                 "cpu_ms_median": round(c_med, 1), "cpu_ms_p95": round(c_p95, 1)})
    print(rows[-1])
speed = pd.DataFrame(rows); speed.to_csv(f"{BASE}/results/s8c_latency.csv", index=False)
print("\nCPU threads used:", torch.get_num_threads()); speed

Mounted at /content/drive


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

{'model': 'minilm_l6', 'params_M': 22.7, 'val_ndcg': np.float64(0.9267), 'gpu_ms_median': np.float64(22.0), 'gpu_ms_p95': np.float64(30.2), 'cpu_ms_median': np.float64(1218.6), 'cpu_ms_p95': np.float64(2639.0)}


...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'model': 'bert_base', 'params_M': 109.5, 'val_ndcg': np.float64(0.9251), 'gpu_ms_median': np.float64(63.7), 'gpu_ms_p95': np.float64(85.0), 'cpu_ms_median': np.float64(7641.6), 'cpu_ms_p95': np.float64(10755.3)}

CPU threads used: 1


,model,params_M,val_ndcg,gpu_ms_median,gpu_ms_p95,cpu_ms_median,cpu_ms_p95
0,minilm_l6,22.7,0.9267,22.0,30.2,1218.6,2639.0
1,minilm_l12,33.4,0.9301,29.9,41.7,2397.5,3348.1
2,distilbert,67.0,0.9209,35.9,45.8,3806.5,5316.5
3,bert_base,109.5,0.9251,63.7,85.0,7641.6,10755.3
